# Campaña factorial HQCNN en Google Colab Pro+ (TASK-20 / TASK-13)

Entorno canónico de entrenamiento del proyecto de grado. Este cuaderno **no implementa** modelos, particiones ni bucles de entrenamiento: importa `src.experiments.campana` del repositorio, que es la única fuente de verdad (TASK-1).

Lo que sí vive aquí, y solo aquí, es el **contrato de rutas hacia Google Drive**. Ningún módulo de `src/` conoce Drive: todas sus rutas se derivan de `ExperimentConfig` con `pathlib`.

## Qué ejecuta este cuaderno

1. Verificación de GPU y dependencias con los pines del `README`.
2. Montaje de Drive, dataset y persistencia de `results/` y `models/`.
3. Higiene del registro: archivado de corridas ajenas a la campaña en CUDA (decisión D3).
4. Sonda de 1 época del HQCNN en CUDA, que cierra la compuerta de presupuesto de TASK-11.
5. Los bloques de la campaña factorial: 60 celdas, HQCNN al 100 % al final.
6. Verificación de integridad y comparación de costo.

## Antes de empezar

- Entorno de ejecución con GPU **T4** (`Entorno de ejecución` → `Cambiar tipo de entorno de ejecución`). El cuello de botella es la simulación del VQC con `parameter-shift`, que se evalúa en CPU: una GPU de gama alta acelera el *backbone* congelado, no el término dominante del costo.
- En Google Drive debe existir un **atajo en Mi unidad** a la carpeta `brain_tumor_mri` (con `Training/` y `Testing/`). Carpeta compartida: [Drive — brain_tumor_mri](https://drive.google.com/drive/folders/1BMroUroMvh8_vLv7lUiy8384_bUARx7u?usp=drive_link). Tras montar Drive, la ruta esperada es `/content/drive/MyDrive/brain_tumor_mri`.
- El repositorio remoto es [frankdaza/uao_ia_cd_proyecto_grado](https://github.com/frankdaza/uao_ia_cd_proyecto_grado). `src/experiments/campana.py` y este cuaderno deben estar en `main` antes de clonar.

> **Nota sobre UV.** Colab es la única excepción documentada al gestor UV del proyecto. En local sigue rigiendo `uv sync` / `uv run`.

## Paso 1 — Verificar la GPU asignada

Colab no garantiza el mismo modelo de GPU entre sesiones. Anotar cuál tocó: si cambia a mitad de campaña, los tiempos por época dejan de ser comparables entre bloques y hay que registrarlo en el estado de ejecución de TASK-13.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## Paso 2 — Contrato de rutas y montaje de Drive

Única celda del proyecto donde aparecen rutas de Drive. Ajustar las constantes a la cuenta propia.

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

# --- Contrato de rutas (ajustar solo si el atajo de Drive tiene otro nombre) ---
# Dataset: atajo en Mi unidad → https://drive.google.com/drive/folders/1BMroUroMvh8_vLv7lUiy8384_bUARx7u
CARPETA_DATASET_DRIVE = Path('/content/drive/MyDrive/brain_tumor_mri')
# Artefactos de campaña (CSV, historiales, pesos): carpeta separada del dataset
RAIZ_DRIVE = Path('/content/drive/MyDrive/tesis_hqcnn')
URL_REPO = 'https://github.com/frankdaza/uao_ia_cd_proyecto_grado.git'
RAIZ_REPO = Path('/content/uao_ia_cd_proyecto_grado')

RAIZ_DRIVE.mkdir(parents=True, exist_ok=True)
assert CARPETA_DATASET_DRIVE.is_dir(), (
    f'No se encontró {CARPETA_DATASET_DRIVE}. '
    'Añade un atajo en Mi unidad a la carpeta brain_tumor_mri de Drive.'
)
print('Drive montado | dataset:', CARPETA_DATASET_DRIVE, '| artefactos:', RAIZ_DRIVE)

## Paso 3 — Clonar o actualizar el repositorio

El repositorio se clona en el disco local del runtime, no en Drive: el I/O de Drive es lento y el árbol de código se lee en cada importación.

In [ ]:
import os
import subprocess

if RAIZ_REPO.exists():
    subprocess.run(['git', '-C', str(RAIZ_REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', URL_REPO, str(RAIZ_REPO)], check=True)

os.chdir(RAIZ_REPO)
print('Directorio de trabajo:', Path.cwd())

## Paso 4 — Dependencias con los pines del `README`

Las mismas versiones del `pyproject.toml`. No instalar «latest»: el combo validado es PyTorch 2.9.1 con PennyLane 0.45; versiones más nuevas de PyTorch pueden romper `TorchLayer` y la regla de cambio de parámetros.

In [ ]:
!pip install pennylane==0.45.1 pennylane-lightning==0.45.0 --quiet
!pip install torch==2.9.1 torchvision==0.24.1 torchaudio==2.9.1 \
    --index-url https://download.pytorch.org/whl/cu128 --quiet
!pip install "numpy>=2.0,<2.3" scikit-learn==1.9.0 "scipy>=1.16,<1.17" \
    statsmodels==0.14.6 "pandas>=2.3,<2.4" "matplotlib>=3.10,<3.11" \
    "seaborn>=0.13,<0.14" "wandb>=0.28,<0.29" "tqdm>=4.67,<5" \
    "pillow>=11,<12" --quiet

**Reiniciar el entorno de ejecución** si pip reemplazó una versión ya importada, y continuar desde aquí (los pasos previos son idempotentes).

In [ ]:
import pennylane as qml
import torch

print('torch', torch.__version__, '| pennylane', qml.__version__)
assert torch.cuda.is_available(), 'Sin CUDA: cambiar el entorno de ejecución a GPU antes de continuar'
print('GPU:', torch.cuda.get_device_name(0))

## Paso 5 — Dataset

El árbol `brain_tumor_mri` se **copia al disco local** del runtime (`data/brain_tumor_mri/`). Entrenar leyendo imagen por imagen desde Drive FUSE domina el tiempo por época.

Antes de continuar, se verifica byte a byte contra `results/dataset_manifest.csv` (TASK-3): si Drive difiere del manifiesto auditado, `splits.json` deja de ser válido y la campaña debe abortar.

In [ ]:
import shutil

import pandas as pd

from src.data.audit import verificar_arbol_contra_manifiesto

DESTINO_DATOS = RAIZ_REPO / 'data' / 'brain_tumor_mri'
RUTA_MANIFIESTO = RAIZ_REPO / 'results' / 'dataset_manifest.csv'

if not any(DESTINO_DATOS.glob('*')):
    print('Copiando dataset desde Drive al disco local (puede tardar varios minutos)...')
    shutil.copytree(CARPETA_DATASET_DRIVE, DESTINO_DATOS, dirs_exist_ok=True)
else:
    print('Dataset local ya presente; se omite la copia.')

manifiesto = pd.read_csv(RUTA_MANIFIESTO)
verificacion = verificar_arbol_contra_manifiesto(DESTINO_DATOS, manifiesto)
print(
    f"Verificación TASK-3: {verificacion['hash_ok']}/{len(manifiesto)} hashes OK | "
    f"faltantes={len(verificacion['faltantes'])} | "
    f"hash_distinto={len(verificacion['hash_distinto'])} | "
    f"extras={len(verificacion['extras'])}"
)
if verificacion['extras'][:5]:
    print('Archivos extra (no bloquean):', verificacion['extras'][:5])
assert not verificacion['faltantes'], verificacion['faltantes'][:5]
assert not verificacion['hash_distinto'], verificacion['hash_distinto'][:5]
assert verificacion['hash_ok'] == len(manifiesto)

print('Clases por partición:')
for particion in sorted(DESTINO_DATOS.glob('*')):
    if particion.is_dir():
        clases = sorted(p.name for p in particion.iterdir() if p.is_dir())
        n_jpg = sum(1 for _ in particion.rglob('*.jpg'))
        print(f'  {particion.name}: {clases} ({n_jpg} .jpg)')

## Paso 6 — Persistencia en Drive

`results/` y `models/` se enlazan a Drive para que un corte de sesión no destruya el CSV, los historiales ni los pesos.

La siembra desde el repositorio ocurre **una sola vez**, cuando la carpeta de Drive está vacía. Si ya hay artefactos allí, Drive es la fuente de verdad: lo contrario sobrescribiría con el CSV vacío del repositorio las celdas ya entrenadas.

In [ ]:
for carpeta in ('results', 'models'):
    destino = RAIZ_DRIVE / carpeta
    destino.mkdir(parents=True, exist_ok=True)
    local = RAIZ_REPO / carpeta

    drive_vacio = not any(destino.rglob('*'))
    if local.is_symlink():
        local.unlink()
    elif local.exists():
        if drive_vacio:
            shutil.copytree(local, destino, dirs_exist_ok=True)
            print(f'{carpeta}/: sembrado en Drive desde el repositorio')
        else:
            print(f'{carpeta}/: Drive ya tiene artefactos, se conserva lo de Drive')
        shutil.rmtree(local)
    local.symlink_to(destino, target_is_directory=True)

!ls -la results/ | head -12

## Paso 7 — Monitoreo con wandb sin conexión

`WANDB_MODE=offline` evita perder corridas cuando la sesión se corta a mitad de un bloque (TASK-4). La sincronización se hace al final, cuando ya existen los directorios locales de la corrida.

In [ ]:
os.environ['WANDB_MODE'] = 'offline'
os.environ['MPLCONFIGDIR'] = '/content/.matplotlib'
print('wandb en modo', os.environ['WANDB_MODE'])

## Paso 8 — Higiene del registro experimental (decisión D3)

Saca de `experiments.csv` toda corrida que no pertenezca a la campaña en CUDA y la conserva como evidencia:

- presupuesto de épocas distinto al del protocolo → `results/pruebas_informales.csv`;
- dispositivo distinto a `cuda` → `results/historico_mps.csv`.

Los historiales van a `results/history_mps/` y las celdas afectadas vuelven a `pendiente`, de modo que la reanudabilidad no las omita. La operación es idempotente: si no hay nada que archivar, no toca nada.

In [ ]:
!python -m src.experiments.campana --archivar-no-cuda

## Paso 9 — Sonda de 1 época del HQCNN en CUDA

Cierra la compuerta de presupuesto que TASK-11 dejó abierta (`decision: no-go`, `sonda_1_epoca_pendiente: true`), medida allí sobre CPU.

La sonda escribe en el CSV como cualquier corrida; la celda siguiente la retira hacia `pruebas_informales.csv`, porque con una sola época no tiene validez inferencial y dejarla en el registro haría que la reanudabilidad omitiera esa celda en la campaña real.

In [ ]:
!python -m src.experiments.campana \
    --modelo hqcnn --fraccion 0.10 --fold 0 --max-epocas 1 --sin-baselines-previas

In [ ]:
import csv

with open('results/experiments.csv', newline='', encoding='utf-8') as archivo:
    for fila in csv.DictReader(archivo):
        if fila['modelo'] == 'hqcnn' and int(fila['epocas']) == 1:
            segundos = float(fila['train_time_s'])
            print(f"Sonda HQCNN en {fila['dispositivo']}: {segundos:.1f} s/época")
            print(f'Aceleración frente a los 388.9 s medidos en CPU: {388.9 / segundos:.1f}x')
            # 20 celdas de HQCNN, con el costo escalando con la fracción de datos.
            factor_fracciones = (0.10 + 0.25 + 0.50 + 1.00) / 0.10
            horas_hqcnn = segundos * 15 * 5 * factor_fracciones / 3600
            print(f'Horas re-extrapoladas del bloque HQCNN: {horas_hqcnn:.1f} h')

!python -m src.experiments.campana --archivar-no-cuda

Con ese número, actualizar a mano `results/selected_hparams.json` (`presupuesto.horas_campana_estimadas`, `presupuesto.decision`, `sonda_1_epoca_pendiente: false`, `dispositivo_campana_real`) y registrar el resultado en el hallazgo `hallazgo:task-20` de la bitácora antes de lanzar la campaña.

## Paso 10 — Bloques de la campaña (TASK-13)

Un bloque por celda de este cuaderno, para poder reanudar exactamente donde se cortó la sesión. La reanudabilidad compara la clave `(modelo, fracción, fold, semilla)`: relanzar un bloque ya terminado no reentrena nada.

**Ningún hiperparámetro cambia entre bloques**: `L = 6`, 15 épocas, semilla 42, lote 32, `lr = 1e-3`. Si algo resultara imprescindible cambiar, hay que repetir el bloque comparable completo, no añadir la celda suelta.

Al terminar cada bloque, actualizar la tabla de estado de ejecución del hallazgo `hallazgo:task-13` en la bitácora. Es registro incremental: no se espera al final de la campaña.

### Bloque 1 — Líneas base al 100 % (revalidación en GPU)

Rehace en CUDA las diez celdas de TASK-12 que estaban medidas en `mps`. Además de dar homogeneidad de hardware al diseño, permite contrastar la exactitud contra valores conocidos (EfficientNet-B0 $\approx$ 0.937, ResNet-50 $\approx$ 0.920) antes de comprometer horas en el bloque híbrido.

In [ ]:
!python -m src.experiments.campana --fraccion 1.00 --modelo efficientnet_b0 --sin-baselines-previas
!python -m src.experiments.campana --fraccion 1.00 --modelo resnet50 --sin-baselines-previas

### Bloque 2 — Fracción del 10 %

In [ ]:
!python -m src.experiments.campana --fraccion 0.10

### Bloque 3 — Fracción del 25 %

In [ ]:
!python -m src.experiments.campana --fraccion 0.25

### Bloque 4 — Fracción del 50 %

In [ ]:
!python -m src.experiments.campana --fraccion 0.50

### Bloque 5 — HQCNN al 100 %

El bloque más caro del proyecto. Va al final por gestión de riesgo: si el presupuesto de cómputo se agota, lo que falta es la celda más costosa y no la mitad del diseño.

In [ ]:
!python -m src.experiments.campana --fraccion 1.00

## Paso 11 — Integridad y costo

Condición de entrada a TASK-14: 60 celdas únicas, sin duplicados y con historial por época completo.

In [ ]:
!python -m src.experiments.campana --verificar

In [ ]:
import pandas as pd

df = pd.read_csv('results/experiments.csv')
print('Filas:', len(df), '| dispositivos:', df['dispositivo'].unique())
print('Horas reales acumuladas:', round(df['train_time_s'].sum() / 3600, 2))
df.groupby(['modelo', 'data_fraction'])['accuracy_val'].agg(['mean', 'std']).round(4)

## Paso 12 — Cierre de la sesión

1. Sincronizar wandb (requiere `wandb login`).
2. Descargar `results/` desde Drive al repositorio local y versionarlo.
3. Actualizar en la bitácora el hallazgo `hallazgo:task-13` con el estado de ejecución y el costo acumulado, y `hallazgo:task-20` con la medición de la sonda.

El análisis posterior (TASK-14, TASK-15, TASK-16) se ejecuta en **CPU**, local con `uv run`: no consume unidades de cómputo.

In [ ]:
!wandb sync --sync-all || echo 'Sin corridas offline pendientes o falta wandb login'